# 00 · Preparar el entorno y los datos

**Python que piensa: agentes de analítica con LangChain y LangGraph**
Sociedad Ecuatoriana de Estadística · Semana de Python

---

Este notebook es el primero que hay que correr. Hace cuatro cosas:

1. Verifica que el entorno tenga lo necesario.
2. Descarga los datos del CMS, si no están ya.
3. Construye la base SQLite ejecutando los scripts de `datos/`.
4. Comprueba que la base responda y recorre las ocho trampas del dato.

No hay nada de LangChain todavía. Esto es la plomería: se corre una vez y no se
vuelve a tocar.

> **Sobre los datos.** Son reclamos médicos **sintéticos** del CMS de Estados
> Unidos. El organismo alteró deliberadamente las relaciones entre variables para
> proteger la privacidad. Sirven para construir y probar herramientas; **no
> admiten inferencia sobre la población real de Medicare.**

## Legibilidad para proyección

Esta celda agranda la tipografía de las salidas. En una sala proyectada, el
tamaño por defecto no se lee desde la cuarta fila.

In [1]:
from IPython.display import HTML, display

display(HTML("""
<style>
  .jp-RenderedText pre, .jp-OutputArea-output pre, div.output_area pre {
      font-size: 15px !important; line-height: 1.5 !important; }
  .jp-RenderedHTMLCommon table { font-size: 15px !important; }
  .jp-InputArea-editor, .CodeMirror { font-size: 15px !important; }
</style>
"""))
print("Tipografía ajustada para proyección.")

Tipografía ajustada para proyección.


## Dónde estamos

Todas las rutas del proyecto son **relativas a la raíz del repositorio**, nunca
absolutas: así el notebook corre igual en otra máquina. Esta celda resuelve la
raíz y la deja en `RAIZ`.

In [2]:
import sys
from pathlib import Path

# La raíz es la carpeta que contiene datos/. Buscamos hacia arriba desde el cwd,
# para que el notebook funcione aunque Jupyter se haya abierto en otro nivel.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos").is_dir()), None)
if RAIZ is None:
    raise SystemExit("No encuentro la carpeta datos/. Abre Jupyter en la raíz del repositorio.")

DATOS = RAIZ / "datos"
BASE = DATOS / "reclamos.db"

# datos/ tiene los scripts del pipeline; hay que poder importarlos.
if str(DATOS) not in sys.path:
    sys.path.insert(0, str(DATOS))

print(f"Raíz del proyecto : {RAIZ}")
print(f"Scripts de datos  : {DATOS}")
print(f"Base de datos     : {BASE}  {'(existe)' if BASE.exists() else '(aún no existe)'}")

Raíz del proyecto : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend
Scripts de datos  : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos
Base de datos     : c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos\reclamos.db  (existe)


## Verificar el entorno

Antes de construir nada, revisamos qué hay y qué falta. La idea es que si algo
no está, se sepa **ahora** y no en el minuto 26 de la charla.

El pipeline de datos solo necesita `sqlite3` y `csv`, que vienen con Python, más
`requests` para la descarga. Lo demás es para los notebooks que siguen.

In [3]:
import importlib.util
import sqlite3

print(f"Python {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    print("  ADVERTENCIA: el proyecto se escribió para 3.11+")
print(f"SQLite {sqlite3.sqlite_version}\n")

# (módulo a importar, paquete en pip, para qué sirve, imprescindible ahora)
PAQUETES = [
    ("requests",     "requests",        "descargar los datos del CMS",      True),
    ("yaml",         "pyyaml",          "leer el diccionario de negocio",   True),
    ("pandas",       "pandas",          "mostrar resultados como tablas",   False),
    ("dotenv",       "python-dotenv",   "leer el .env con las claves",      False),
    ("langchain",    "langchain",       "notebook 02",                      False),
    ("langgraph",    "langgraph",       "notebook 03",                      False),
    ("langsmith",    "langsmith",       "notebook 04",                      False),
    ("fastapi",      "fastapi",         "notebook 05",                      False),
    ("ipywidgets",   "ipywidgets",      "la caja de preguntas en vivo",     False),
]

faltan_ahora, faltan_despues = [], []
for modulo, paquete, para_que, imprescindible in PAQUETES:
    hay = importlib.util.find_spec(modulo) is not None
    marca = "  ok  " if hay else ("FALTA " if imprescindible else " --   ")
    print(f"{marca} {paquete:<16} {para_que}")
    if not hay:
        (faltan_ahora if imprescindible else faltan_despues).append(paquete)

print()
if faltan_ahora:
    print("Falta lo imprescindible. Instala y vuelve a correr esta celda:")
    print(f"  pip install {' '.join(faltan_ahora)}")
elif faltan_despues:
    print("Lo imprescindible está. Para los notebooks que siguen te faltará:")
    print(f"  pip install {' '.join(faltan_despues)}")
    print("  (o directamente:  pip install -r requirements.txt)")
else:
    print("Entorno completo.")

Python 3.13.10
SQLite 3.50.4

  ok   requests         descargar los datos del CMS
  ok   pyyaml           leer el diccionario de negocio
  ok   pandas           mostrar resultados como tablas
  ok   python-dotenv    leer el .env con las claves
  ok   langchain        notebook 02
  ok   langgraph        notebook 03
  ok   langsmith        notebook 04
  ok   fastapi          notebook 05
  ok   ipywidgets       la caja de preguntas en vivo

Entorno completo.


## Las credenciales del modelo

El `.env` no hace falta para construir la base, pero sí desde el notebook 01.
Lo revisamos ahora para no descubrir que falta más adelante.

**El `.env` nunca va al repositorio.** Está en el `.gitignore`. Lo que sí va es
`.env.example`, con los nombres de las variables y sin los valores.

In [4]:
env = RAIZ / ".env"
if not env.exists():
    print("No hay .env todavía. No importa para esta celda ni para construir la base.")
    print("Antes del notebook 01: copia .env.example a .env y pon tus claves.")
else:
    # Leemos solo los nombres de las variables. Los valores no se imprimen nunca:
    # esta salida se commitea al repositorio.
    nombres = [l.split("=", 1)[0].strip() for l in env.read_text(encoding="utf-8").splitlines()
               if l.strip() and not l.strip().startswith("#") and "=" in l]
    print(f".env encontrado, con {len(nombres)} variables definidas:")
    for n in nombres:
        print(f"  {n}")
    print("\n(Los valores no se imprimen: la salida de este notebook va a git.)")

.env encontrado, con 9 variables definidas:
  OPENAI_API_KEY
  ANTHROPIC_API_KEY
  TRAZAS
  LANGSMITH_TRACING
  LANGSMITH_ENDPOINT
  LANGSMITH_API_KEY
  LANGSMITH_PROJECT
  AZURE_OPENAI_BASE_URL
  AZURE_OPENAI_API_KEY

(Los valores no se imprimen: la salida de este notebook va a git.)


## Paso 1 · Descargar los datos del CMS

Tres archivos, 40 MB comprimidos y 193 MB descomprimidos:

| Archivo | Se convierte en |
|---|---|
| Beneficiary Summary 2008 | `afiliados` |
| Inpatient Claims 2008-2010 | `hospitalario` |
| Outpatient Claims 2008-2010 | `ambulatorio` |

La celda es **idempotente**: si los CSV ya están, no vuelve a bajar nada.

Si tu red bloquea `cms.gov` —pasa en redes corporativas y entornos con lista
blanca— la celda te lo dirá. En ese caso baja los tres `.zip` a mano desde la
[página de la muestra 1](https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-1)
y descomprímelos en `datos/extraidos/`.

In [5]:
import descargar

ya_estan = [DATOS / "extraidos" / csv for _, csv, _ in descargar.ARCHIVOS.values()]
if all(p.exists() for p in ya_estan):
    print("Los tres CSV ya están descomprimidos. No hay nada que descargar.\n")
    for p in ya_estan:
        print(f"  {p.name:<52} {p.stat().st_size/1_048_576:>7.1f} MB")
else:
    import os
    anterior = Path.cwd()
    os.chdir(DATOS)          # los scripts trabajan con rutas relativas a datos/
    try:
        if descargar.verificar():
            descargar.descargar()
        else:
            print("\nAlgún enlace del CMS no respondió. Mira el mensaje de arriba.")
    finally:
        os.chdir(anterior)

Los tres CSV ya están descomprimidos. No hay nada que descargar.

  DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv        13.9 MB
  DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv        15.9 MB
  DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv      154.3 MB


## Paso 2 · Construir la base

`preparar.py` hace el trabajo real:

- Traduce las columnas crípticas del CMS a nombres de negocio en español.
- Traduce también **los valores**: sexo, raza, estado de residencia.
- Convierte las fechas y las parte en `anio`, `mes` y `dia`.
- Normaliza los indicadores de condición crónica, que en el origen vienen
  invertidos (1 = sí, 2 = no).
- Clasifica los 11 897 códigos de diagnóstico en los 19 capítulos de la CIE-9.
- Crea los índices y **verifica las ocho trampas** con cifras reales.

Tarda unos 45 segundos y deja `datos/reclamos.db`. La salida de abajo es la
verificación: si algún número no coincide con el README, algo se rompió.

In [6]:
import os
import preparar

anterior = Path.cwd()
os.chdir(DATOS)
try:
    preparar.main(entrada="extraidos", salida="reclamos.db")
finally:
    os.chdir(anterior)

Construyendo reclamos.db desde extraidos/ ...


  cargando afiliados      ... 116,352 filas
  cargando hospitalario   ... 66,773 filas
  cargando ambulatorio    ... 790,790 filas
  cargando capitulos_cie9 ... 19 filas
  derivando diagnosticos ... 2,611,067 filas
  creando índices ...

--- Volúmenes ---
  afiliados         116,352 filas
  hospitalario       66,773 filas
  ambulatorio       790,790 filas
  diagnosticos    2,611,067 filas

--- Trampa 1: crónicos (1=sí, 2=no en el origen) ---
  afiliados con diabetes: 44,060 de 116,352 (37.9%)   [el codebook del CMS dice 37.96%]

--- Trampa 2: no existe 'costo total' ---
  pagado por el asegurador :   639,260,180.00
  deducible del afiliado   :    68,280,716.00
  coaseguro del afiliado   :     6,011,500.00
  pagado por un tercero    :    26,635,700.00
  costo del episodio       :   740,188,096.00  (+15.8%)

--- Trampa 3: montos negativos y en cero ---
  hospitalario       55 negativos (mín -8,000.00),   2,160 en cero
  ambulatorio     2,566 negativos (mín -100.00),  30,105 en cero

--- 

## Paso 3 · Comprobar que la base responde

Hasta aquí confiamos en lo que imprimió el script. Ahora consultamos la base
nosotros, como lo hará el agente.

In [7]:
import sqlite3

con = sqlite3.connect(BASE)

print("Tablas:\n")
for (nombre,) in con.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    n = con.execute(f"SELECT count(*) FROM {nombre}").fetchone()[0]
    print(f"  {nombre:<18} {n:>12,} filas")

print(f"\nTamaño del archivo: {BASE.stat().st_size/1_048_576:.1f} MB")

Tablas:

  afiliados               116,352 filas
  ambulatorio             790,790 filas
  capitulos_cie9               20 filas
  diagnosticos          2,611,067 filas
  hospitalario             66,773 filas

Tamaño del archivo: 556.3 MB


### La pregunta fácil

Es la primera que va a responder el agente en el notebook 02. Aquí la hacemos a
mano, para tener el número correcto con el que comparar después.

In [8]:
sql = """
SELECT sum(monto_pagado) AS pagado,
       count(DISTINCT id_reclamo) AS reclamos
FROM hospitalario
WHERE anio = 2009
"""
pagado, reclamos = con.execute(sql).fetchone()
print(f"¿Cuánto se pagó en hospitalización en 2009?\n")
print(f"  Pagado por el asegurador : {pagado:>16,.2f}")
print(f"  Reclamos                 : {reclamos:>16,}")
print(f"  Costo promedio           : {pagado/reclamos:>16,.2f}")

¿Cuánto se pagó en hospitalización en 2009?

  Pagado por el asegurador :   244,810,270.00
  Reclamos                 :           25,231
  Costo promedio           :         9,702.76


### Lo que el agente va a tener que entender

Las columnas están en español y los valores también. Esto es lo que vuelve
viable que un modelo escriba SQL correcto sin adivinar.

In [9]:
for consulta, titulo in [
    ("SELECT sexo, count(*) FROM afiliados GROUP BY sexo ORDER BY 2 DESC",
     "sexo"),
    ("SELECT raza, count(*) FROM afiliados GROUP BY raza ORDER BY 2 DESC",
     "raza"),
    ("SELECT estado, count(*) c FROM afiliados GROUP BY estado ORDER BY c DESC LIMIT 5",
     "estado (los 5 primeros de 52)"),
]:
    print(f"{titulo}:")
    for valor, n in con.execute(consulta):
        print(f"   {valor:<24} {n:>8,}")
    print()

sexo:
   Femenino                   64,347
   Masculino                  52,005

raza:
   Blanca                     96,349
   Negra                      12,343
   Otras                       4,931
   Hispana                     2,729

estado (los 5 primeros de 52):
   California                 10,224
   Florida                     7,745
   Texas                       6,703
   Nueva York                  6,510
   Pensilvania                 5,199



### Los diagnósticos, en lenguaje de negocio

Los 11 897 códigos CIE-9 de la base son ilegibles para una persona. La columna
`grupo_enfermedad` los agrupa en los 19 capítulos de la clasificación, en
español.

**Esta es la diferencia entre un agente que acierta y uno que comunica.**
"El alza está en 41401" no le dice nada a nadie; "el alza está en enfermedades
del aparato circulatorio" sí.

Ojo con el `WHERE orden = 1`: sin ese filtro cada reclamo se cuenta hasta diez
veces, una por cada diagnóstico, y el monto se multiplica.

In [10]:
sql = """
SELECT d.grupo_enfermedad,
       sum(h.monto_pagado) AS pagado,
       count(*)            AS reclamos
FROM diagnosticos d
JOIN hospitalario h ON h.id_reclamo = d.id_reclamo
WHERE d.ambito = 'hospitalario' AND d.orden = 1
GROUP BY d.grupo_enfermedad
ORDER BY pagado DESC
LIMIT 8
"""
print(f"{'Grupo de enfermedad':<58} {'Pagado':>16} {'Reclamos':>10}")
print("-" * 86)
for grupo, pagado, reclamos in con.execute(sql):
    print(f"{grupo:<58} {pagado:>16,.0f} {reclamos:>10,}")

Grupo de enfermedad                                                  Pagado   Reclamos
--------------------------------------------------------------------------------------
Enfermedades del aparato circulatorio                           165,127,000     15,901
Enfermedades del aparato respiratorio                            75,982,000      8,453
Lesiones y envenenamientos                                       67,375,420      6,050
Enfermedades del aparato digestivo                               55,095,130      6,475
Enfermedades del sistema osteomuscular y tejido conjuntivo       53,467,840      4,730
Enfermedades infecciosas y parasitarias                          44,855,820      3,375
Neoplasias                                                       36,521,010      2,920
Factores que influyen en el estado de salud                      32,754,350      2,157


## Paso 4 · Las ocho trampas, una por una

Este es el corazón pedagógico de la charla. **El agente no falla por el modelo:
falla por no saber qué significa cada columna.**

Cada trampa de abajo es una forma en que una consulta correcta en SQL da una
respuesta equivocada en negocio. Todas están documentadas con su verificación en
`datos/diccionario.yaml`, que es lo que el agente va a leer antes de escribir
SQL.

In [11]:
q = lambda s: con.execute(s).fetchone()

print("1 · LOS INDICADORES CRÓNICOS VENÍAN INVERTIDOS")
n, d = q("SELECT count(*), sum(cronico_diabetes) FROM afiliados")
print(f"    Afiliados con diabetes: {d:,} de {n:,} = {d/n:.1%}")
print(f"    El codebook del CMS dice 37.96%. Si saliera 62%, la conversión")
print(f"    está mal: en el origen 1 = sí y 2 = no, no 1/0.\n")

print("2 · NO EXISTE UNA COLUMNA DE COSTO TOTAL")
pag, ded, coa, ter = q("""SELECT sum(monto_pagado), sum(deducible),
                          sum(coaseguro), sum(monto_pagado_tercero)
                          FROM hospitalario""")
tot = pag + ded + coa + ter
print(f"    Pagó el asegurador  : {pag:>16,.2f}   <- 'costo' por defecto")
print(f"    Deducible           : {ded:>16,.2f}")
print(f"    Coaseguro           : {coa:>16,.2f}")
print(f"    Pagó un tercero     : {ter:>16,.2f}")
print(f"    Costo del episodio  : {tot:>16,.2f}   <- {tot/pag-1:+.1%}")
print(f"    Son dos preguntas distintas. El agente debe desambiguar.\n")

print("3 · HAY MONTOS NEGATIVOS Y EN CERO, Y SON VÁLIDOS")
for t in ("hospitalario", "ambulatorio"):
    neg, mn = q(f"SELECT count(*), min(monto_pagado) FROM {t} WHERE monto_pagado < 0")
    cero = q(f"SELECT count(*) FROM {t} WHERE monto_pagado = 0")[0]
    print(f"    {t:<14} {neg:>6,} negativos (mín {mn:,.0f}), {cero:>7,} en cero")
print(f"    Un WHERE monto > 0 puesto por reflejo los borra.\n")

print("4 · EL PERIODO REAL NO ES EL DEL NOMBRE DEL ARCHIVO")
for t in ("hospitalario", "ambulatorio"):
    desde, hasta = q(f"SELECT min(fecha_inicio), max(fecha_inicio) FROM {t}")
    fuera = q(f"SELECT count(*) FROM {t} WHERE anio NOT BETWEEN 2008 AND 2010")[0]
    print(f"    {t:<14} {desde} a {hasta} — {fuera:,} reclamos fuera de 2008-2010")
print(f"    El archivo se llama '2008_to_2010' y trae 2007.\n")

print("5 · CONTAR FILAS NO ES CONTAR RECLAMOS")
for t in ("hospitalario", "ambulatorio"):
    filas, recl = q(f"SELECT count(*), count(DISTINCT id_reclamo) FROM {t}")
    print(f"    {t:<14} {filas:>9,} filas -> {recl:>9,} reclamos "
          f"(sobran {filas-recl:,})")
print(f"    Un reclamo largo viene partido en hasta dos segmentos.\n")

print("6 · UN CUARTO DE LOS AFILIADOS NO TIENE NINGÚN RECLAMO")
sin = q("""SELECT count(*) FROM afiliados a WHERE NOT EXISTS
           (SELECT 1 FROM hospitalario h WHERE h.id_afiliado = a.id_afiliado)
           AND NOT EXISTS
           (SELECT 1 FROM ambulatorio o WHERE o.id_afiliado = a.id_afiliado)""")[0]
tot_afi = q("SELECT count(*) FROM afiliados")[0]
print(f"    {sin:,} de {tot_afi:,} = {sin/tot_afi:.1%}")
print(f"    Un INNER JOIN los desaparece e infla el gasto per cápita.\n")

print("7 · LOS MONTOS ANUALES DE afiliados SON SOLO DE 2008")
anual = q("SELECT sum(pago_anual_hospitalario) FROM afiliados")[0]
r2008 = q("SELECT sum(monto_pagado) FROM hospitalario WHERE anio = 2008")[0]
print(f"    Suma anual en afiliados : {anual:>16,.2f}  (solo 2008)")
print(f"    Reclamos de 2008        : {r2008:>16,.2f}")
print(f"    Las tablas de reclamos cubren tres años. No son comparables.\n")

print("8 · 'OTHER' NO ES UN CÓDIGO DE DIAGNÓSTICO")
n_other = q("SELECT count(*) FROM diagnosticos WHERE codigo = 'OTHER'")[0]
sin_cap = q("SELECT count(*) FROM diagnosticos WHERE capitulo IS NULL")[0]
print(f"    Aparece {n_other:,} veces. Es un marcador que dejó el CMS.")
print(f"    Tiene capítulo 99, 'Sin clasificar', para que no desaparezca")
print(f"    en un GROUP BY. Diagnósticos sin capítulo: {sin_cap}.")

1 · LOS INDICADORES CRÓNICOS VENÍAN INVERTIDOS
    Afiliados con diabetes: 44,060 de 116,352 = 37.9%
    El codebook del CMS dice 37.96%. Si saliera 62%, la conversión
    está mal: en el origen 1 = sí y 2 = no, no 1/0.

2 · NO EXISTE UNA COLUMNA DE COSTO TOTAL
    Pagó el asegurador  :   639,260,180.00   <- 'costo' por defecto
    Deducible           :    68,280,716.00
    Coaseguro           :     6,011,500.00
    Pagó un tercero     :    26,635,700.00
    Costo del episodio  :   740,188,096.00   <- +15.8%
    Son dos preguntas distintas. El agente debe desambiguar.

3 · HAY MONTOS NEGATIVOS Y EN CERO, Y SON VÁLIDOS
    hospitalario       55 negativos (mín -8,000),   2,160 en cero
    ambulatorio     2,566 negativos (mín -100),  30,105 en cero
    Un WHERE monto > 0 puesto por reflejo los borra.

4 · EL PERIODO REAL NO ES EL DEL NOMBRE DEL ARCHIVO
    hospitalario   2007-11-27 a 2010-12-30 — 224 reclamos fuera de 2008-2010
    ambulatorio    2007-12-12 a 2010-12-31 — 312 reclamos fue

## Listo

La base está construida y verificada. De aquí en adelante ya no hay que volver a
`datos/`.

```python
import sqlite3
con = sqlite3.connect("datos/reclamos.db")
```

**Lo que sigue:** `02_langchain/` — la primera herramienta, para que un modelo
pueda consultar esta base en lenguaje natural.

---

### Opcional: las descripciones de los códigos CIE-9

Los 11 897 códigos de diagnóstico quedaron sin descripción a propósito: no hay
catálogo oficial en español, y traducir ese volumen de terminología clínica sin
fuente sería inventar significados médicos.

Para hablar de diagnósticos ya tenemos `grupo_enfermedad` (19 capítulos en
español) y `categoria` (los tres primeros dígitos). Con eso alcanza para la
charla.

Si además quieres el texto oficial código por código, `datos/enriquecer.py` lo
baja del catálogo de la Biblioteca Nacional de Medicina de Estados Unidos. Está
**en inglés**, y requiere red hacia `clinicaltables.nlm.nih.gov`:

```python
import enriquecer
enriquecer.main(BASE)
```

Es opcional. El proyecto funciona sin eso.

---

# Parte 2 · La sábana: de cuatro tablas a una

Hasta aquí la base está **desagregada**: afiliados, hospitalario, ambulatorio y
diagnósticos, cada una con su grano. Es la forma correcta de guardar datos,
pero obliga al agente a escribir uniones, y **cada unión es una oportunidad de
equivocarse**: contar segmentos en lugar de reclamos, multiplicar montos por
diez diagnósticos, perder dinero sin fecha.

Por eso armamos un modelo **agregado**: una sola tabla, `sabana`, con **una fila
por reclamo**, hospitalario y ambulatorio juntos, y todo ya resuelto. El agente
solo filtra y agrupa.

Los dos modelos conviven en la misma base, cada uno con su diccionario:

| Modelo | Tablas | Diccionario | Quién lo usa |
|---|---|---|---|
| Desagregado | 4 tablas + catálogo | `datos/diccionario.yaml` | Documentación del pipeline |
| Agregado | `sabana` | `datos/diccionario_sabana.yaml` | **El agente** |

## Configuración de la vista

Los resultados se muestran como tablas de pandas, con separador de miles y todas
las columnas visibles: las tablas del CMS son anchas.

In [12]:
import sqlite3
import pandas as pd

pd.options.display.max_columns = None
pd.options.display.width = 250
pd.options.display.float_format = "{:,.2f}".format

con = sqlite3.connect(BASE)
print(f"Conectado a {BASE.name}")

Conectado a reclamos.db


## Cómo venía la base · afiliados

Una fila por asegurado: demografía, meses de cobertura, condiciones crónicas y
montos anuales (solo de 2008, trampa 7). No trae ningún reclamo.

In [13]:
pd.read_sql("SELECT * FROM afiliados LIMIT 5", con)

,id_afiliado,fecha_nacimiento,fecha_defuncion,sexo,raza,enfermedad_renal_terminal,estado,codigo_estado,codigo_condado,meses_cobertura_hospitalaria,meses_cobertura_medica,meses_cobertura_hmo,meses_cobertura_farmacia,cronico_alzheimer,cronico_insuficiencia_cardiaca,cronico_renal,cronico_cancer,cronico_epoc,cronico_depresion,cronico_diabetes,cronico_cardiopatia_isquemica,cronico_osteoporosis,cronico_artritis,cronico_acv,pago_anual_hospitalario,copago_anual_hospitalario,pago_tercero_hospitalario,pago_anual_ambulatorio,copago_anual_ambulatorio,pago_tercero_ambulatorio,pago_anual_honorarios,copago_anual_honorarios,pago_tercero_honorarios
0,00013D2EFD8E45D1,1923-05-01,None,Masculino,Blanca,0,Misuri,26,950,12,12,12,12,0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,50.00,10.00,0.00,0.00,0.00,0.00
1,00016F745862898F,1943-01-01,None,Masculino,Blanca,0,Pensilvania,39,230,12,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,700.00,240.00,0.00
2,0001FDD721E223DC,1936-09-01,None,Femenino,Blanca,0,Pensilvania,39,280,12,12,0,12,0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,00021CA6FF03E670,1941-06-01,None,Masculino,Hispana,0,Colorado,06,290,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,00024B3D2352D2D0,1936-08-01,None,Masculino,Blanca,0,Wisconsin,52,590,12,12,0,0,0,0,0,0,0,0,0,0,1,0,0,0.00,0.00,0.00,30.00,40.00,0.00,220.00,80.00,0.00


## Cómo venía la base · hospitalario

Una fila por **segmento** de reclamo de internación, con diez columnas de
diagnóstico y seis de procedimiento.

In [14]:
pd.read_sql("SELECT * FROM hospitalario LIMIT 5", con)

,id_afiliado,id_reclamo,segmento,fecha_inicio,fecha_fin,fecha_ingreso,fecha_alta,dias_estancia,id_prestador,medico_tratante,medico_operante,monto_pagado,monto_pagado_tercero,deducible,coaseguro,deducible_sangre,per_diem,grupo_drg,diagnostico_ingreso,diagnostico_1,diagnostico_2,diagnostico_3,diagnostico_4,diagnostico_5,diagnostico_6,diagnostico_7,diagnostico_8,diagnostico_9,diagnostico_10,procedimiento_1,procedimiento_2,procedimiento_3,procedimiento_4,procedimiento_5,procedimiento_6,anio,mes,dia
0,00013D2EFD8E45D1,196661176988405,1,2010-03-12,2010-03-13,2010-03-12,2010-03-13,1,2600GD,3139083564,NaN,"4,000.00",0.00,"1,100.00",0.00,0.00,0.00,217,4580,7802,78820,V4501,4280,2720,4019,V4502,73300,E9330,None,NaN,NaN,NaN,None,None,None,2010,3,12
1,00016F745862898F,196201177000368,1,2009-04-12,2009-04-18,2009-04-12,2009-04-18,6,3900MB,6476809087,NaN,"26,000.00",0.00,"1,068.00",0.00,0.00,0.00,201,7866,1970,4019,5853,7843,2768,71590,2724,19889,5849,None,NaN,NaN,NaN,None,None,None,2009,4,12
2,00016F745862898F,196661177015632,1,2009-08-31,2009-09-02,2009-08-31,2009-09-02,2,3900HM,0611998537,0611998537,"5,000.00",0.00,"1,068.00",0.00,0.00,0.00,750,6186,6186,2948,56400,NaN,NaN,NaN,NaN,NaN,NaN,None,7092,6186,V5866,None,None,None,2009,8,31
3,00016F745862898F,196091176981058,1,2009-09-17,2009-09-20,2009-09-17,2009-09-20,3,3913XU,4971602784,NaN,"5,000.00",0.00,"1,068.00",0.00,0.00,0.00,883,29590,29623,30390,71690,34590,V1581,32723,NaN,NaN,NaN,None,NaN,NaN,NaN,None,None,None,2009,9,17
4,00016F745862898F,196261176983265,1,2010-06-26,2010-07-01,2010-06-26,2010-07-01,5,3900MB,6408400473,1960859579,"16,000.00",0.00,"1,100.00",0.00,0.00,0.00,983,5849,3569,4019,3542,V8801,78820,2639,7840,7856,4271,None,NaN,E8889,NaN,None,None,None,2010,6,26


### Un reclamo partido en dos

Algunos reclamos vienen en dos segmentos. **El segundo no trae fecha ni
diagnóstico**, solo dinero. Si se filtra `WHERE anio = 2009` sobre esta tabla,
ese dinero desaparece sin aviso: es la **trampa 9**.

In [15]:
pd.read_sql("""
    SELECT id_reclamo, segmento, fecha_inicio, anio, monto_pagado, diagnostico_1
    FROM hospitalario
    WHERE id_reclamo IN (SELECT id_reclamo FROM hospitalario
                         WHERE segmento = 2 LIMIT 2)
    ORDER BY id_reclamo, segmento
""", con)

,id_reclamo,segmento,fecha_inicio,anio,monto_pagado,diagnostico_1
0,196531176970382,1,2009-01-24,"2,009.00","8,000.00",40390
1,196531176970382,2,NaN,NaN,"8,000.00",NaN
2,196901176966106,1,2008-07-07,"2,008.00","8,000.00",57410
3,196901176966106,2,NaN,NaN,"11,000.00",NaN


## Cómo venía la base · ambulatorio

El mismo esquema que hospitalario, sin días de estancia ni grupo DRG. Es la
tabla más grande: casi 800 mil segmentos.

In [16]:
pd.read_sql("SELECT * FROM ambulatorio LIMIT 5", con)

,id_afiliado,id_reclamo,segmento,fecha_inicio,fecha_fin,id_prestador,medico_tratante,medico_operante,monto_pagado,monto_pagado_tercero,deducible,coaseguro,deducible_sangre,diagnostico_ingreso,diagnostico_1,diagnostico_2,diagnostico_3,diagnostico_4,diagnostico_5,diagnostico_6,diagnostico_7,diagnostico_8,diagnostico_9,diagnostico_10,procedimiento_1,procedimiento_2,procedimiento_3,procedimiento_4,procedimiento_5,procedimiento_6,anio,mes,dia
0,00013D2EFD8E45D1,542192281063886,1,2008-09-04,2008-09-04,2600RA,4824842417,None,50.00,0.00,0.00,10.00,0.00,V5883,V5841,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,2008,9,4
1,00016F745862898F,542272281166593,1,2009-06-02,2009-06-02,3901GS,2963419753,None,30.00,0.00,0.00,0.00,0.00,NaN,V5832,V5861,2724,3182,V5869,42731,None,None,None,None,None,None,None,None,None,None,2009,6,2
2,00016F745862898F,542282281644416,1,2009-06-23,2009-06-23,3939PG,5737807753,None,30.00,0.00,0.00,70.00,0.00,NaN,9594,E9174,4019,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,2009,6,23
3,0001FDD721E223DC,542642281250669,1,2009-10-11,2009-10-11,3902NU,1233847710,None,30.00,0.00,0.00,50.00,0.00,56409,78943,V5866,V1272,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,2009,10,11
4,00024B3D2352D2D0,542242281386963,1,2008-07-12,2008-07-12,5200TV,9688809345,None,30.00,0.00,0.00,40.00,0.00,60021,6009,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,None,None,None,None,2008,7,12


## Las tablas unidas

Esto es lo que el agente tendría que escribir **cada vez** para responder algo
tan simple como "costo por grupo de enfermedad y sexo": tres tablas, dos
uniones, y el filtro `orden = 1` que, si se olvida, multiplica el monto.

In [17]:
pd.read_sql("""
    SELECT h.id_reclamo, h.segmento, h.anio, h.monto_pagado,
           a.sexo, a.raza, a.estado, a.fecha_nacimiento,
           d.codigo AS diagnostico, d.grupo_enfermedad
    FROM hospitalario h
    JOIN afiliados    a ON a.id_afiliado = h.id_afiliado
    JOIN diagnosticos d ON d.id_reclamo  = h.id_reclamo
                       AND d.ambito = 'hospitalario'
                       AND d.orden  = 1
    LIMIT 5
""", con)

,id_reclamo,segmento,anio,monto_pagado,sexo,raza,estado,fecha_nacimiento,diagnostico,grupo_enfermedad
0,196661176988405,1,2010,"4,000.00",Masculino,Blanca,Misuri,1923-05-01,7802,"Síntomas, signos y estados mal definidos"
1,196201177000368,1,2009,"26,000.00",Masculino,Blanca,Pensilvania,1943-01-01,1970,Neoplasias
2,196661177015632,1,2009,"5,000.00",Masculino,Blanca,Pensilvania,1943-01-01,6186,Enfermedades del aparato genitourinario
3,196091176981058,1,2009,"5,000.00",Masculino,Blanca,Pensilvania,1943-01-01,29623,Trastornos mentales
4,196261176983265,1,2010,"16,000.00",Masculino,Blanca,Pensilvania,1943-01-01,3569,Enfermedades del sistema nervioso y de los sen...


## Construir la sábana

`datos/sabana.py` hace el trabajo, en SQL, en unos 30 segundos:

- **Una fila por reclamo.** Los segmentos se consolidan: los montos se suman y
  fecha, diagnóstico y médico salen del segmento 1.
- **Hospitalario y ambulatorio juntos**, distinguidos por la columna `ambito`.
- **Del afiliado:** sexo, raza, estado, condado, edad a la fecha del reclamo y
  banda de edad.
- **Del diagnóstico principal:** categoría y grupo de enfermedad en español.
- **`monto_total`** = pagado + deducible + coaseguro + pagado por tercero. Los
  cuatro montos desagregados se conservan.
- **Exclusiones:** 2007 y 2010, y los reclamos con monto pagado negativo.

Al final cuadra la sábana contra las tablas de origen: mismos reclamos y mismo
dinero.

In [18]:
import sabana

sabana.main(BASE)

Armando la sábana en c:\Users\ricar\Documents\SEE\Charla2026Sep\PythonWeekend\datos\reclamos.db ...

--- Cuadre contra el origen (2008-2009, sin negativos) ---
  Hospitalario  origen    52,866 reclamos   503,576,710.00  | sábana    52,866   503,576,710.00  OK
  Ambulatorio   origen   603,281 reclamos   172,633,780.00  | sábana   603,281   172,633,780.00  OK
  reclamos repetidos en la sábana: 0

--- Totales por año y ámbito ---
  2008 Ambulatorio     281,971 reclamos  pagado   78,449,490.00  total  106,026,240.00
  2008 Hospitalario     27,656 reclamos  pagado  258,063,920.00  total  299,344,636.00
  2009 Ambulatorio     321,310 reclamos  pagado   94,184,290.00  total  125,977,830.00
  2009 Hospitalario     25,210 reclamos  pagado  245,512,790.00  total  284,055,506.00

--- Bandas de edad ---
  21-30      5,275
  31-40     14,084
  41-50     28,234
  51-60     45,242
  61-70    165,914
  71+      397,398

Listo: tabla sabana con 656,147 reclamos y 25 columnas.


## Qué se retiró y por qué

Se trabaja solo con **2008 y 2009**, los dos años completos:

- **2007** trae apenas noviembre y diciembre: es un residuo, no un año.
- **2010** tiene los doce meses, pero los reclamos caen mes a mes hasta que
  diciembre tiene el 14 % de enero en hospitalario: el año aún no estaba
  cerrado cuando el CMS armó la muestra.
- **Los montos negativos** son ajustes y reversiones. Se excluyen. Los montos en
  cero se quedan: son reclamos válidos.
- **278 reclamos ambulatorios sin fecha**: cada uno trae solo su segundo segmento, que
  no tiene fecha, así que no hay forma de ubicarlos en un año.

Cada reclamo cae en un solo motivo, en el orden de la tabla.

In [19]:
retirados = pd.read_sql("""
    WITH reclamos AS (
        SELECT id_reclamo, max(id_afiliado) AS id_afiliado,
               max(anio) AS anio, sum(monto_pagado) AS monto
        FROM hospitalario GROUP BY id_reclamo
        UNION ALL
        SELECT id_reclamo, max(id_afiliado), max(anio), sum(monto_pagado)
        FROM ambulatorio GROUP BY id_reclamo
    )
    SELECT CASE WHEN anio IS NULL THEN '0. Sin fecha (solo segundo segmento)'
                WHEN anio = 2007 THEN '1. Año 2007 (solo nov-dic)'
                WHEN anio = 2010 THEN '2. Año 2010 (incompleto)'
                WHEN monto < 0   THEN '3. Monto pagado negativo'
                ELSE                  '4. Queda en la sábana' END AS motivo,
           count(*)                    AS reclamos,
           sum(monto)                  AS monto_pagado,
           count(DISTINCT id_afiliado) AS afiliados_con_reclamos_aqui
    FROM reclamos
    GROUP BY motivo
    ORDER BY motivo
""", con)
retirados

,motivo,reclamos,monto_pagado,afiliados_con_reclamos_aqui
0,0. Sin fecha (solo segundo segmento),278,"392,320.00",147
1,1. Año 2007 (solo nov-dic),536,"3,114,030.00",504
2,2. Año 2010 (incompleto),187543,"184,156,770.00",58514
3,3. Monto pagado negativo,2016,"-88,720.00",1972
4,4. Queda en la sábana,656147,"676,210,490.00",79692


### Los afiliados que no llegan a la sábana

La sábana solo tiene personas con al menos un reclamo en 2008-2009. Quedan
fuera dos grupos: quienes **nunca** tuvieron un reclamo (trampa 6) y quienes
solo los tuvieron en 2007, 2010, negativos o sin fecha. Por eso **en la sábana no se puede calcular el
gasto per cápita**: el denominador no está completo.

In [20]:
total      = con.execute("SELECT count(*) FROM afiliados").fetchone()[0]
en_sabana  = con.execute("SELECT count(DISTINCT id_afiliado) FROM sabana").fetchone()[0]
sin_nada   = con.execute("""
    SELECT count(*) FROM afiliados
    WHERE id_afiliado NOT IN (SELECT id_afiliado FROM hospitalario)
      AND id_afiliado NOT IN (SELECT id_afiliado FROM ambulatorio)
""").fetchone()[0]

pd.DataFrame({
    "afiliados": [total, en_sabana, sin_nada, total - en_sabana - sin_nada],
}, index=["Total en la base",
          "En la sábana (reclamos en 2008-2009)",
          "Fuera: nunca tuvieron un reclamo",
          "Fuera: reclamos solo en 2007, 2010, negativos o sin fecha"])

,afiliados
Total en la base,116352
En la sábana (reclamos en 2008-2009),79692
Fuera: nunca tuvieron un reclamo,29614
"Fuera: reclamos solo en 2007, 2010, negativos o sin fecha",7046


## La sábana por dentro

25 columnas, una fila por reclamo. Ya no hay segmentos, ni uniones, ni columnas
de diagnóstico repetidas diez veces.

In [21]:
pd.read_sql("SELECT * FROM sabana LIMIT 5", con)

,id_reclamo,id_afiliado,ambito,fecha_inicio,anio,mes,monto_pagado,deducible,coaseguro,monto_pagado_tercero,monto_total,sexo,raza,edad,banda_edad,estado,codigo_condado,diagnostico_principal,categoria,grupo_enfermedad,procedimiento_principal,medico_tratante,id_prestador,grupo_drg,dias_estancia
0,196731176974632,00597F3605020D7E,Hospitalario,2008-01-18,2008,1,"10,000.00","1,024.00",0.00,0.00,"11,024.00",Masculino,Blanca,84,71+,Kansas,860,71536,715,Enfermedades del sistema osteomuscular y tejid...,8154,2860740867,1700KK,464,3
1,196641177016461,00945FDC75DB9AE9,Hospitalario,2008-01-30,2008,1,"10,000.00","1,024.00",0.00,0.00,"11,024.00",Femenino,Blanca,64,61-70,Virginia Occidental,180,1460,146,Neoplasias,4041,3563988001,2100RT,149,2
2,196351176988422,011C85D52630E6FB,Hospitalario,2008-01-16,2008,1,"9,000.00",0.00,0.00,0.00,"9,000.00",Femenino,Blanca,52,51-60,Arizona,060,57400,574,Enfermedades del aparato digestivo,5123,2995346199,0300DD,421,5
3,196201177018154,011C85D52630E6FB,Hospitalario,2008-01-19,2008,1,"3,000.00","1,024.00",0.00,0.00,"4,024.00",Femenino,Blanca,52,51-60,Arizona,060,5225,522,Enfermedades del aparato digestivo,NaN,9627739511,0300QR,159,3
4,196701177021816,012AD144DA8A37C6,Hospitalario,2008-01-30,2008,1,"28,000.00","1,024.00","2,000.00",0.00,"31,024.00",Masculino,Blanca,72,71+,Texas,681,03842,038,Enfermedades infecciosas y parasitarias,5474,3383300189,4502HM,858,11


### Totales por año y ámbito

La diferencia entre `monto_pagado` y `monto_total` es la trampa 2: no existe un
único "costo". El agente tiene que declarar cuál usó.

In [22]:
pd.read_sql("""
    SELECT anio, ambito,
           count(*)          AS reclamos,
           sum(monto_pagado) AS monto_pagado,
           sum(monto_total)  AS monto_total
    FROM sabana
    GROUP BY anio, ambito
    ORDER BY anio, ambito
""", con)

,anio,ambito,reclamos,monto_pagado,monto_total
0,2008,Ambulatorio,281971,"78,449,490.00","106,026,240.00"
1,2008,Hospitalario,27656,"258,063,920.00","299,344,636.00"
2,2009,Ambulatorio,321310,"94,184,290.00","125,977,830.00"
3,2009,Hospitalario,25210,"245,512,790.00","284,055,506.00"


### Bandas de edad

Las bandas son **0-5, 6-10 y de 10 en 10 hasta 71+**.

> **Recordatorio:** las bandas de 0 a 20 años existen pero quedan **vacías**. La
> edad mínima en los reclamos es 24: Medicare cubre a mayores de 65 y a personas
> con discapacidad. No es un error del pipeline. Y la banda 71+ concentra más de
> la mitad de los reclamos.

In [23]:
bandas = ["00-05", "06-10", "11-20", "21-30", "31-40", "41-50", "51-60", "61-70", "71+"]

por_banda = pd.read_sql("""
    SELECT banda_edad, count(*) AS reclamos, sum(monto_pagado) AS monto_pagado
    FROM sabana
    GROUP BY banda_edad
""", con).set_index("banda_edad").reindex(bandas, fill_value=0)

por_banda["% reclamos"] = (100 * por_banda["reclamos"] / por_banda["reclamos"].sum()).round(1)
por_banda

,reclamos,monto_pagado,% reclamos
banda_edad,,,
00-05,0,0.00,0.00
06-10,0,0.00,0.00
11-20,0,0.00,0.00
21-30,5275,"4,972,730.00",0.80
31-40,14084,"13,234,910.00",2.10
41-50,28234,"28,829,720.00",4.30
51-60,45242,"47,381,200.00",6.90
61-70,165914,"161,444,500.00",25.30
71+,397398,"420,347,430.00",60.60


## La pregunta fácil, antes y después

"¿Cuánto se pagó en hospitalización en 2009?". Sobre la tabla segmentada, el
`WHERE anio = 2009` pierde los segundos segmentos, que no tienen fecha. Sobre la
sábana no hay nada que perder: cada fila es un reclamo completo.

La diferencia también incluye los reclamos negativos de 2009, que la sábana
excluye.

In [24]:
segmentada = con.execute(
    "SELECT sum(monto_pagado) FROM hospitalario WHERE anio = 2009").fetchone()[0]
en_sabana = con.execute(
    "SELECT sum(monto_pagado) FROM sabana WHERE ambito = 'Hospitalario' AND anio = 2009").fetchone()[0]

pd.DataFrame({"pagado_2009": [segmentada, en_sabana, en_sabana - segmentada]},
             index=["Tabla segmentada (WHERE anio = 2009)",
                    "Sábana (una fila por reclamo)",
                    "Diferencia"])

,pagado_2009
Tabla segmentada (WHERE anio = 2009),"244,810,270.00"
Sábana (una fila por reclamo),"245,512,790.00"
Diferencia,"702,520.00"


## Listo · Parte 2

La base tiene ahora los dos modelos. Desde el notebook 02 en adelante el agente
trabaja **solo con la sábana** y lee `datos/diccionario_sabana.yaml`: columnas,
valores exactos, trampas vigentes y diez consultas de ejemplo con su resultado
verificado.